In [ ]:
import sys; sys.path.insert(0, '..')
from solver.grid import Grid
from solver.mapf import Drone, MAPFSolver
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

In [ ]:
grid = Grid(rows=8, cols=8)
drones = [
    Drone(0, (0,0), (7,7)),
    Drone(1, (7,0), (0,7)),
    Drone(2, (0,7), (7,0)),
    Drone(3, (3,0), (3,7)),
]
sol = MAPFSolver(grid, drones, time_limit_s=30).solve()
print(f"Status: {sol.status} | Makespan: {sol.makespan} | Solve: {sol.solve_time_ms:.0f}ms")

In [ ]:
COLORS = ['#3b82f6','#ef4444','#22c55e','#f59e0b','#a855f7',
          '#06b6d4','#ec4899','#84cc16','#fb923c','#e11d48']

def animate_solution(grid, sol, drones, obstacles=None, nofly=None, interval=300):
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_xlim(-0.5, grid.cols - 0.5)
    ax.set_ylim(-0.5, grid.rows - 0.5)
    ax.set_facecolor('#0f172a')
    ax.set_aspect('equal')
    ax.invert_yaxis()

    for r in range(grid.rows + 1):
        ax.axhline(r - 0.5, color='#1e3a5f', lw=0.5)
    for c in range(grid.cols + 1):
        ax.axvline(c - 0.5, color='#1e3a5f', lw=0.5)

    for (r, c) in (obstacles or []):
        ax.add_patch(plt.Rectangle((c-.5, r-.5), 1, 1, color='#334155'))

    for (r, c) in (nofly or []):
        ax.add_patch(plt.Rectangle((c-.5, r-.5), 1, 1, color='#ef4444', alpha=0.3))

    for i, d in enumerate(drones):
        col = COLORS[i % len(COLORS)]
        ax.plot(d.start[1], d.start[0], 'o', ms=10, color=col, alpha=0.4)
        ax.plot(d.goal[1], d.goal[0], '*', ms=12, color=col)

    dots = [ax.plot([], [], 'o', ms=12, color=COLORS[i % len(COLORS)])[0]
            for i in range(len(drones))]

    def update(frame):
        for i, d in enumerate(drones):
            path = sol.paths[d.id]
            pos = path[min(frame, len(path)-1)]
            dots[i].set_data([pos[1]], [pos[0]])
        ax.set_title(f"t={frame}/{sol.makespan}  makespan={sol.makespan}  "
                     f"solve={sol.solve_time_ms:.0f}ms", color='white')
        return dots

    fig.patch.set_facecolor('#0f172a')
    ax.tick_params(colors='#64748b')
    return animation.FuncAnimation(fig, update, frames=sol.makespan+1,
                                   interval=interval, blit=True)

anim = animate_solution(grid, sol, drones)
from IPython.display import HTML
HTML(anim.to_jshtml())

In [ ]:
import time

results = []
for n in [2, 3, 4, 5, 6, 7, 8]:
    g = Grid(rows=8, cols=8)
    ds = [Drone(i, (0, i*2 % 8), (7, (7 - i*2) % 8)) for i in range(n)]
    t0 = time.time()
    s = MAPFSolver(g, ds, time_limit_s=30).solve()
    elapsed = time.time() - t0
    results.append({"n": n, "status": s.status, "ms": s.solve_time_ms, "makespan": s.makespan})
    print(f"n={n}: {s.status} makespan={s.makespan} in {s.solve_time_ms:.0f}ms")

ns = [r["n"] for r in results if r["status"] in ("optimal","feasible")]
ms = [r["ms"] for r in results if r["status"] in ("optimal","feasible")]
plt.figure(facecolor='#0f172a')
ax = plt.gca()
ax.set_facecolor('#0f172a')
ax.plot(ns, ms, 'o-', color='#3b82f6')
ax.set_xlabel("Number of drones", color='#94a3b8')
ax.set_ylabel("Solve time (ms)", color='#94a3b8')
ax.set_title("CP-SAT solve time vs agent count (8×8)", color='white')
ax.tick_params(colors='#94a3b8')
plt.tight_layout(); plt.show()